# NCS-O*NET 매핑 프로젝트: 데이터 전처리 파이프라인
## Cell 1: 라이브러리 import 및 경로 설정

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
from deep_translator import GoogleTranslator
import warnings
warnings.filterwarnings('ignore')

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
ONET_DIR = RAW_DIR / "db_30_2_excel"
PROCESSED_DIR.mkdir(exist_ok=True)

print("라이브러리 로드 완료")
print(f"RAW_DIR     : {RAW_DIR.resolve()}")
print(f"PROCESSED_DIR: {PROCESSED_DIR.resolve()}")
print(f"ONET_DIR    : {ONET_DIR.resolve()}")

## Cell 2: NCS 데이터 로딩 및 사업관리 필터링

In [ ]:
# NCS_DB.xlsx 로드
ncs_raw = pd.read_excel(RAW_DIR / "NCS_DB.xlsx", engine="openpyxl")
print(f"[원본] shape: {ncs_raw.shape}")
print(f"[원본] columns:\n  {ncs_raw.columns.tolist()}\n")

# 중분류 컬럼 자동 탐지: '중분류'와 '명'이 모두 포함된 컬럼
mid_col = next(
    (c for c in ncs_raw.columns if "중분류" in c and "명" in c),
    None
)
if mid_col is None:
    raise ValueError("중분류명 컬럼을 찾을 수 없습니다. 컬럼 목록을 확인하세요.")
print(f"중분류명 컬럼 감지: '{mid_col}'")
print(f"중분류 고유값: {ncs_raw[mid_col].unique().tolist()}\n")

# 사업관리 필터링
ncs_filtered = ncs_raw[ncs_raw[mid_col].astype(str).str.contains("사업관리", na=False)].copy()
print(f"[필터링 후] shape: {ncs_filtered.shape}")

# 소분류 컬럼 자동 탐지
sub_col = next(
    (c for c in ncs_filtered.columns if "소분류" in c and "명" in c),
    None
)
print(f"\n소분류명 컬럼 감지: '{sub_col}'")
print("소분류별 행 수:")
print(ncs_filtered[sub_col].value_counts().to_string())

# 결측값 처리
ncs_filtered = ncs_filtered.fillna("")
print(f"\n결측값 fillna('') 처리 완료")

## Cell 3: NCS 텍스트 합산

In [ ]:
# 텍스트 합산 대상 컬럼 우선순위 매핑 (실제 컬럼명 자동 탐지)
TARGET_KEYWORDS = ["소분류", "세분류", "능력단위명칭", "능력단위요소명"]

def find_col(df, keyword):
    """컬럼명에 keyword가 포함된 첫 번째 컬럼 반환"""
    candidates = [c for c in df.columns if keyword in c]
    if not candidates:
        return None
    # '명칭' or '명'이 있는 걸 우선, 아니면 첫 번째
    named = [c for c in candidates if c.endswith("명칭") or (c.endswith("명") and "번호" not in c and "코드" not in c)]
    return named[0] if named else candidates[0]

col_sub    = find_col(ncs_filtered, "소분류") or sub_col   # 소분류코드명
col_detail = find_col(ncs_filtered, "세분류")               # 세분류코드명
col_unit   = find_col(ncs_filtered, "능력단위명칭")          # 능력단위명칭
col_elem   = find_col(ncs_filtered, "능력단위요소명")        # 능력단위요소명

text_cols = [c for c in [col_sub, col_detail, col_unit, col_elem] if c is not None]
print(f"텍스트 합산 대상 컬럼: {text_cols}\n")

# 행별 combined_text 생성
ncs_filtered["combined_text"] = ncs_filtered[text_cols].astype(str).apply(
    lambda row: " ".join(v for v in row if v.strip()), axis=1
)

# 소분류 단위 그룹핑
ncs_grouped = (
    ncs_filtered.groupby(col_sub)["combined_text"]
    .apply(lambda x: " ".join(x))
    .reset_index()
)
ncs_grouped.columns = ["소분류코드명", "combined_text"]

print("=== 소분류별 텍스트 길이 ===")
for _, row in ncs_grouped.iterrows():
    print(f"  [{row['소분류코드명']}]  {len(row['combined_text']):,} 자")

# 전체 사업관리 텍스트 1개로 합산
ncs_full_text = " ".join(ncs_grouped["combined_text"].tolist())
print(f"\n전체 ncs_full_text 길이: {len(ncs_full_text):,} 자")

# 저장
save_path = PROCESSED_DIR / "ncs_사업관리_processed.csv"
ncs_grouped.to_csv(save_path, index=False, encoding="utf-8-sig")
print(f"\n저장 완료: {save_path}")

## Cell 4: O*NET 데이터 로딩

In [ ]:
# Occupation Data 로드
onet_occ = pd.read_excel(ONET_DIR / "Occupation Data.xlsx", engine="openpyxl")
onet_occ.columns = onet_occ.columns.str.strip()
print(f"[Occupation Data] shape: {onet_occ.shape}")
print(f"columns: {onet_occ.columns.tolist()}\n")

# Task Statements 로드
onet_tasks = pd.read_excel(ONET_DIR / "Task Statements.xlsx", engine="openpyxl")
onet_tasks.columns = onet_tasks.columns.str.strip()
print(f"[Task Statements] shape: {onet_tasks.shape}")
print(f"columns: {onet_tasks.columns.tolist()}\n")

# SOC Code 기준 Task 텍스트 합산
soc_col  = "O*NET-SOC Code"
task_col = next((c for c in onet_tasks.columns if "task" in c.lower() and "id" not in c.lower() and "type" not in c.lower()), None)
print(f"Task 텍스트 컬럼 감지: '{task_col}'")

task_combined = (
    onet_tasks.groupby(soc_col)[task_col]
    .apply(lambda x: " ".join(x.dropna().astype(str)))
    .reset_index()
)
task_combined.columns = [soc_col, "task_text"]
print(f"\ntask_combined shape: {task_combined.shape}")

# Occupation Data와 merge
onet_merged = onet_occ.merge(task_combined, on=soc_col, how="left")

# final_text 구성
desc_col = next((c for c in onet_merged.columns if "description" in c.lower()), None)
if desc_col:
    print(f"Description 컬럼 발견: '{desc_col}' → task_text + Description 합산")
    onet_merged["final_text"] = (
        onet_merged["task_text"].fillna("") + " " + onet_merged[desc_col].fillna("")
    ).str.strip()
else:
    print("Description 컬럼 없음 → task_text만 사용")
    onet_merged["final_text"] = onet_merged["task_text"].fillna("")

print(f"\n[onet_merged] shape: {onet_merged.shape}")
print("\n=== 샘플 2건 ===")
print(onet_merged[[soc_col, "Title", "final_text"]].head(2).to_string())

# 저장
save_path = PROCESSED_DIR / "onet_processed.csv"
onet_merged.to_csv(save_path, index=False, encoding="utf-8-sig")
print(f"\n저장 완료: {save_path}")

## Cell 5: NCS 텍스트 영어 번역

In [ ]:
CHUNK_SIZE = 500  # GoogleTranslator 안전 청크 크기 (최대 5000자이나 안전하게 500자 단위)

def translate_text(text: str, src: str = "ko", tgt: str = "en") -> str:
    """텍스트를 CHUNK_SIZE 단위로 나눠 번역 후 합산."""
    if not text or not text.strip():
        return text
    translator = GoogleTranslator(source=src, target=tgt)
    chunks = [text[i:i + CHUNK_SIZE] for i in range(0, len(text), CHUNK_SIZE)]
    translated_chunks = []
    for chunk in chunks:
        try:
            translated_chunks.append(translator.translate(chunk))
        except Exception as e:
            print(f"  [경고] 청크 번역 실패 (원문 유지): {e}")
            translated_chunks.append(chunk)
    return " ".join(translated_chunks)


# ── 전체 합산 텍스트(ncs_full_text) 번역 ──────────────────────────────
print(f"번역 전 텍스트 길이: {len(ncs_full_text):,} 자")
print(f"예상 청크 수: {(len(ncs_full_text) // CHUNK_SIZE) + 1}")
print("번역 시작...")

ncs_full_text_en = translate_text(ncs_full_text)
print(f"번역 후 텍스트 길이: {len(ncs_full_text_en):,} 자")

# txt 파일 저장
txt_path = PROCESSED_DIR / "ncs_translated.txt"
txt_path.write_text(ncs_full_text_en, encoding="utf-8-sig")
print(f"저장 완료: {txt_path}\n")

# ── 소분류별 번역 ──────────────────────────────────────────────────────
print("소분류별 번역 시작...")
ncs_grouped_translated = ncs_grouped.copy()
translated_texts = []
for _, row in tqdm(ncs_grouped_translated.iterrows(), total=len(ncs_grouped_translated), desc="소분류 번역"):
    translated_texts.append(translate_text(row["combined_text"]))

ncs_grouped_translated["combined_text_en"] = translated_texts

# CSV 저장
csv_path = PROCESSED_DIR / "ncs_사업관리_translated.csv"
ncs_grouped_translated.to_csv(csv_path, index=False, encoding="utf-8-sig")
print(f"\n저장 완료: {csv_path}")
print("\n=== 소분류별 번역 결과 길이 ===")
for _, row in ncs_grouped_translated.iterrows():
    print(f"  [{row['소분류코드명']}]  {len(row['combined_text_en']):,} 자")

## Cell 6: 전처리 결과 최종 요약

In [ ]:
print("=" * 60)
print("  전처리 결과 최종 요약")
print("=" * 60)

# NCS 통계
n_sub      = ncs_grouped["소분류코드명"].nunique()
n_unit_col = find_col(ncs_filtered, "능력단위명칭")
n_units    = ncs_filtered[n_unit_col].nunique() if n_unit_col else "N/A"

print(f"\n[NCS 사업관리]")
print(f"  소분류 수           : {n_sub}")
print(f"  전체 능력단위 수    : {n_units}")
print(f"  전체 데이터 행 수   : {len(ncs_filtered):,}")

# O*NET 통계
print(f"\n[O*NET]")
print(f"  전체 직업 수        : {len(onet_occ):,}")
tasks_per_occ = onet_tasks.groupby("O*NET-SOC Code")["Task ID"].count()
print(f"  직업당 평균 Task 수 : {tasks_per_occ.mean():.2f}")

# 번역 샘플
print(f"\n[번역 텍스트 샘플 (첫 300자)]")
print(ncs_full_text_en[:300])

# 저장 파일 목록
print(f"\n[저장된 파일 목록]")
for p in sorted(PROCESSED_DIR.glob("*")):
    size_kb = p.stat().st_size / 1024
    print(f"  {p.name:<45} ({size_kb:,.1f} KB)  →  {p}")

print("\n" + "=" * 60)
print("  전처리 파이프라인 완료")
print("=" * 60)